# Decoder Locality Analysis

## Goal

We edit token IDs inside a contiguous **token-space** patch and decode back to pixels.

The question is:

> when we change tokens only inside a chosen token patch, does the decoded image change mostly inside the corresponding pixel patch?

This notebook uses the **new patch-sweep H2 export format**, where each source image contains:

- multiple **patch fractions** in one run
- multiple **token edit modes** for each patch fraction
- one clean reconstruction plus one edited reconstruction per `(patch fraction, mode)`

## Current H2 Record Format

Each JSONL row corresponds to one source image and one decoder-locality run.

### Core fields

- `token_edit_modes`
  List of edit strategies, e.g. `"random_uniform"`, `"closest"`, `"farthest"`, `"orthogonal"`.

- `token_grid_hw = [H_tok, W_tok]`
  Spatial size of the latent token grid.

- `patch_fraction_sweep`
  Fractions requested in the sweep, e.g. `[0.10, 0.25, 0.50, 0.75]`.

- `patches_by_fraction`
  Mapping from fraction label to patch metadata. For each fraction label such as `"10"` or `"25"`:
  - `target_fraction`
  - `actual_fraction`
  - `patch_side_tok`
  - `patch_bbox_tok = [j0, i0, j1, i1]`
  - `patch_bbox_px = [x0, y0, x1, y1]`

- `indices_clean`
  Flattened clean token grid.

- `indices_edit_by_fraction_and_mode`
  Nested mapping:
  `fraction_label -> mode -> flattened edited token grid`

### Images on disk

For each sample directory under `images/<class>/<image_id>/`:

- `0_original.png`
- `1_recon_clean.png`
- `*_recon_token_edit_patch{fraction_label}_{mode}.png`

Examples:
- `2_recon_token_edit_patch10_random_uniform.png`
- `7_recon_token_edit_patch25_closest.png`
- `13_recon_token_edit_patch50_orthogonal.png`

In [ ]:
from __future__ import annotations

import json
import random
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd
from PIL import Image

from tqdm.auto import tqdm


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

JUPYTER_ROOT = Path("/exp/")

EXPERIMENT_SPECS = {
    "llamagen_imagenet_val": JUPYTER_ROOT / "1773180544_robustness_dataset_llamagen_h2_patch_token_edit_decoder_patchsweep10-25-50-75_seed0",
    "vqgan_imagenet_val": JUPYTER_ROOT / "1773180821_robustness_dataset_vqgan_h2_patch_token_edit_decoder_patchsweep10-25-50-75_seed0",
    "llamagen_imagenetv2": JUPYTER_ROOT / "1774520394_robustness_dataset_llamagen_h2_patch_token_edit_decoder_patchsweep10-25-50-75_seed0",
    "vqgan_imagenetv2": JUPYTER_ROOT / "1774519746_robustness_dataset_vqgan_h2_patch_token_edit_decoder_patchsweep10-25-50-75_seed0",
    "llamagen_sketch": JUPYTER_ROOT / "1773825918_robustness_dataset_llamagen_h2_patch_token_edit_decoder_patchsweep10-25-50-75_seed0",
    "vqgan_sketch": JUPYTER_ROOT / "1773825917_robustness_dataset_vqgan_h2_patch_token_edit_decoder_patchsweep10-25-50-75_seed0",
}


MODE_ORDER = ["closest", "orthogonal", "random_uniform", "farthest"]
FRACTION_ORDER = ["10", "25", "50", "75"]
EDIT_RE = re.compile(
    r"^\d+_recon_token_edit_patch(\d+)_(random_uniform|closest|farthest|orthogonal)\.png$"
)

## Helpers

In [ ]:
@dataclass
class Experiment:
    name: str
    root: Path
    images_dir: Path
    meta_files: list[Path]
    model: str
    dataset: str


def load_img_float01(path: Path) -> np.ndarray:
    return np.asarray(Image.open(path).convert("RGB"), dtype=np.float32) / 255.0


def sorted_meta_files(root: Path) -> list[Path]:
    def key(path: Path) -> tuple[int, str]:
        m = re.search(r"metadata_part_(\d+)\.jsonl$", path.name)
        return (int(m.group(1)), path.name) if m else (10**9, path.name)
    return sorted(root.glob("metadata_part_*.jsonl"), key=key)


def build_experiment(name: str, root: Path) -> Experiment:
    if "llamagen" in name:
        model = "llamagen"
    elif "vqgan" in name:
        model = "vqgan"
    else:
        raise ValueError(f"Could not infer model from experiment name: {name}")

    if "imagenetv2" in name:
        dataset = "imagenetv2"
    elif "sketch" in name:
        dataset = "imagenet_sketch"
    else:
        dataset = "imagenet_val"

    images_dir = root / "images"
    meta_files = sorted_meta_files(root)

    assert root.exists(), f"Missing experiment root: {root}"
    assert images_dir.exists(), f"Missing images dir: {images_dir}"
    assert meta_files, f"No metadata_part_*.jsonl files found in: {root}"

    return Experiment(
        name=name,
        root=root,
        images_dir=images_dir,
        meta_files=meta_files,
        model=model,
        dataset=dataset,
    )


EXPERIMENTS = {
    name: build_experiment(name, root)
    for name, root in EXPERIMENT_SPECS.items()
}

for exp in EXPERIMENTS.values():
    print(f"{exp.name:24s} | model={exp.model:8s} | dataset={exp.dataset:12s} | meta_parts={len(exp.meta_files)}")


In [ ]:
# def delta_map_l2(clean: np.ndarray, edit: np.ndarray) -> np.ndarray:
#     diff = edit - clean
#     return np.sqrt(np.sum(diff * diff, axis=-1))


def delta_map_rmse(clean: np.ndarray, edit: np.ndarray) -> np.ndarray:
    diff = edit - clean
    return np.sqrt(np.mean(diff * diff, axis=-1))


def show_heatmap(ax, heatmap, title="", bbox_px=None, cmap="magma", vmin=None, vmax=None):
    ax.imshow(heatmap, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=10)
    ax.axis("off")
    if bbox_px is not None:
        add_bbox(ax, bbox_px, color="cyan", linewidth=2)


def show_text_panel(ax, text):
    ax.axis("off")
    ax.text(
        0.5, 0.5, text,
        ha="center",
        va="center",
        fontsize=11,
        fontweight="bold",
        transform=ax.transAxes,
    )


In [ ]:
def add_bbox(ax, bbox_px, *, color="cyan", linewidth=2):
    x0, y0, x1, y1 = map(int, bbox_px)
    rect = patches.Rectangle(
        (x0, y0),
        x1 - x0,
        y1 - y0,
        linewidth=linewidth,
        edgecolor=color,
        facecolor="none",
    )
    ax.add_patch(rect)


def show_image(ax, image, title="", bbox_px=None, bbox_color="cyan"):
    ax.imshow(image)
    ax.set_title(title, fontsize=11)
    ax.axis("off")
    if bbox_px is not None:
        add_bbox(ax, bbox_px, color=bbox_color)

## Visulization

In [ ]:
_METADATA_CACHE = {}

def find_metadata_row(exp_name: str, image_id: str) -> dict:
    cache_key = (exp_name, image_id)
    if cache_key in _METADATA_CACHE:
        return _METADATA_CACHE[cache_key]

    exp = EXPERIMENTS[exp_name]

    for meta_path in exp.meta_files:
        with open(meta_path, "r") as f:
            for line in f:
                row = json.loads(line)

                values = [str(v) for v in row.values() if isinstance(v, (str, int, float))]
                blob = " ".join(values)

                if image_id in blob or image_id.split("/", 1)[-1] in blob:
                    _METADATA_CACHE[cache_key] = row
                    return row

    raise KeyError(f"No metadata row found for {exp_name} / {image_id}")

In [ ]:

def build_block_for_image_id(exp_name: str, image_id: str) -> pd.DataFrame:
    exp = EXPERIMENTS[exp_name]

    class_name, sample_id = image_id.split("/", 1)
    sample_dir = exp.images_dir / class_name / sample_id
    assert sample_dir.exists(), f"Missing sample dir: {sample_dir}"

    meta = find_metadata_row(exp_name, image_id)

    orig_path = sample_dir / "0_original.png"
    clean_path = sample_dir / "1_recon_clean.png"

    assert orig_path.exists(), f"Missing original image: {orig_path}"
    assert clean_path.exists(), f"Missing clean image: {clean_path}"

    patches_by_fraction = meta["patches_by_fraction"]

    rows = []
    for p in sorted(sample_dir.iterdir()):
        if not p.is_file():
            continue

        m = EDIT_RE.match(p.name)
        if not m:
            continue

        frac = str(m.group(1))
        mode = str(m.group(2))

        frac_meta = patches_by_fraction[frac]
        bbox_px = frac_meta["patch_bbox_px"]

        rows.append({
            "dataset": exp.dataset,
            "image_id": image_id,
            "orig_path": str(orig_path),
            "clean_path": str(clean_path),
            "edit_path": str(p),
            "fraction_label": frac,
            "mode": mode,
            "bbox_px": bbox_px,
            "patch_bbox_px": bbox_px,
            "original_path": meta.get("original_path"),
        })

    block = pd.DataFrame(rows)
    assert len(block), f"No edit rows found in {sample_dir}"

    required_cols = [
        "fraction_label",
        "mode",
        "orig_path",
        "clean_path",
        "edit_path",
        "bbox_px",
    ]
    missing = [c for c in required_cols if c not in block.columns]
    assert not missing, f"Missing required columns in block: {missing}"

    return block

In [ ]:
## Visualization

N_CLASSES_TO_VISUALIZE = 1
N_PAIRED_SAMPLES_PER_CLASS = 2

DATASET_COMPARE_PAIRS = {
    "imagenet_val": ("llamagen_imagenet_val", "vqgan_imagenet_val"),
    "imagenetv2": ("llamagen_imagenetv2", "vqgan_imagenetv2"),
    "imagenet_sketch": ("llamagen_sketch", "vqgan_sketch"),
}

def list_class_names(images_dir: Path) -> list[str]:
    return sorted(p.name for p in images_dir.iterdir() if p.is_dir())

def list_sample_dir_names(images_dir: Path, class_name: str) -> list[str]:
    class_dir = images_dir / class_name
    return sorted(
        p.name
        for p in class_dir.iterdir()
        if p.is_dir() and not p.name.startswith(".")
    )

# same class across all 3 datasets
class_sets = []
for dataset, (ll_exp_name, vq_exp_name) in DATASET_COMPARE_PAIRS.items():
    ll_exp = EXPERIMENTS[ll_exp_name]
    vq_exp = EXPERIMENTS[vq_exp_name]

    ll_classes = set(list_class_names(ll_exp.images_dir))
    vq_classes = set(list_class_names(vq_exp.images_dir))

    class_sets.append(ll_classes & vq_classes)

shared_classes_across_all = sorted(set.intersection(*class_sets))
assert shared_classes_across_all, "No class shared across all datasets"

selected_classes = shared_classes_across_all[:N_CLASSES_TO_VISUALIZE]

paired_image_ids_by_dataset = {}

for dataset, (ll_exp_name, vq_exp_name) in DATASET_COMPARE_PAIRS.items():
    ll_exp = EXPERIMENTS[ll_exp_name]
    vq_exp = EXPERIMENTS[vq_exp_name]

    paired_image_ids_by_dataset[dataset] = {}

    for cls in selected_classes:
        ll_sample_dirs = set(list_sample_dir_names(ll_exp.images_dir, cls))
        vq_sample_dirs = set(list_sample_dir_names(vq_exp.images_dir, cls))

        shared_sample_dirs = sorted(ll_sample_dirs & vq_sample_dirs)
        assert len(shared_sample_dirs) >= N_PAIRED_SAMPLES_PER_CLASS, (
            f"Not enough shared sample dirs for {dataset} / {cls}"
        )

        paired_image_ids_by_dataset[dataset][cls] = [
            f"{cls}/{sample_dir}"
            for sample_dir in shared_sample_dirs[:N_PAIRED_SAMPLES_PER_CLASS]
        ]

print("Selected classes:", selected_classes)
for dataset, class_map in paired_image_ids_by_dataset.items():
    print(f"\n[{dataset}]")
    for cls, image_ids in class_map.items():
        print(f"  {cls}")
        for image_id in image_ids:
            print("   ", image_id)

In [ ]:
paired_blocks_by_dataset = {}

for dataset, (ll_exp_name, vq_exp_name) in DATASET_COMPARE_PAIRS.items():
    packs = []

    for class_name, image_ids in paired_image_ids_by_dataset[dataset].items():
        for image_id in image_ids:
            ll_block = build_block_for_image_id(ll_exp_name, image_id)
            vq_block = build_block_for_image_id(vq_exp_name, image_id)

            print(
                dataset,
                image_id,
                "| ll cols:", ll_block.columns.tolist(),
                "| vq cols:", vq_block.columns.tolist(),
            )

            packs.append(
                {
                    "class_name": class_name,
                    "image_id": image_id,
                    "llamagen": ll_block,
                    "vqgan": vq_block,
                }
            )

    paired_blocks_by_dataset[dataset] = packs

In [ ]:
def plot_fraction_blocks_comparison(
    block_ll: pd.DataFrame,
    block_vq: pd.DataFrame,
    *,
    heatmap_kind: str = "rmse",
):
    from matplotlib.gridspec import GridSpec

    fractions = [
        f for f in FRACTION_ORDER
        if f in set(block_ll["fraction_label"].astype(str))
        and f in set(block_vq["fraction_label"].astype(str))
    ]
    modes = [
        m for m in MODE_ORDER
        if m in set(block_ll["mode"].astype(str))
        and m in set(block_vq["mode"].astype(str))
    ]

    assert fractions, "No shared fractions found"
    assert modes, "No shared modes found"

    orig = load_img_float01(Path(block_ll["orig_path"].iloc[0]))
    clean_ll = load_img_float01(Path(block_ll["clean_path"].iloc[0]))
    clean_vq = load_img_float01(Path(block_vq["clean_path"].iloc[0]))

    if heatmap_kind == "rmse":
        delta_fn = delta_map_rmse
        heatmap_label = "delta rmse"
    elif heatmap_kind == "l2":
        delta_fn = delta_map_l2
        heatmap_label = "delta l2"
    else:
        raise ValueError(f"Unknown heatmap_kind: {heatmap_kind}")

    heatmaps_ll = {}
    heatmaps_vq = {}
    vmax = 0.0

    for frac in fractions:
        row_ll = block_ll[block_ll["fraction_label"].astype(str) == frac]
        row_vq = block_vq[block_vq["fraction_label"].astype(str) == frac]

        for mode in modes:
            one_ll = row_ll[row_ll["mode"].astype(str) == mode]
            one_vq = row_vq[row_vq["mode"].astype(str) == mode]

            if len(one_ll):
                edit_ll = load_img_float01(Path(one_ll["edit_path"].iloc[0]))
                d_ll = delta_fn(clean_ll, edit_ll)
                heatmaps_ll[(frac, mode)] = d_ll
                vmax = max(vmax, float(d_ll.max()))

            if len(one_vq):
                edit_vq = load_img_float01(Path(one_vq["edit_path"].iloc[0]))
                d_vq = delta_fn(clean_vq, edit_vq)
                heatmaps_vq[(frac, mode)] = d_vq
                vmax = max(vmax, float(d_vq.max()))

    vmax = max(vmax, 1e-8)

    # Layout:
    # 2 top rows
    # then for each fraction:
    #   4 content rows + 1 thin separator row
    ncols = 1 + len(modes)
    nrows = 2 + len(fractions) * 5 - 1

    height_ratios = [1.15, 1.15]
    for i, _ in enumerate(fractions):
        height_ratios.extend([1.35, 1.10, 1.35, 1.10])
        if i != len(fractions) - 1:
            height_ratios.append(0.12)

    fig = plt.figure(figsize=(4.9 * ncols, 3.9 * sum(height_ratios)))
    gs = GridSpec(
        nrows=nrows,
        ncols=ncols,
        figure=fig,
        height_ratios=height_ratios,
        hspace=0.18,
        wspace=0.08,
    )

    title = f"{block_ll['dataset'].iloc[0]} | {block_ll['image_id'].iloc[0]}"
    fig.suptitle(title, fontsize=17, y=0.995)

    # Top reference rows, centered with spans
    ax_orig = fig.add_subplot(gs[0, 1:4])
    show_image(ax_orig, orig, title="original")

    ax_clean_ll = fig.add_subplot(gs[1, 1:3])
    show_image(ax_clean_ll, clean_ll, title="llamagen clean")

    ax_clean_vq = fig.add_subplot(gs[1, 3:5])
    show_image(ax_clean_vq, clean_vq, title="vqgan clean")

    row_ptr = 2

    for fi, frac in enumerate(fractions):
        row_ll = block_ll[block_ll["fraction_label"].astype(str) == frac].copy()
        row_vq = block_vq[block_vq["fraction_label"].astype(str) == frac].copy()
        bbox_px = row_ll["bbox_px"].iloc[0]

        # Fraction block title above the block, not on a separator line through content
        y_text = fig.add_subplot(gs[row_ptr, :]).get_position().y1 + 0.004
        plt.delaxes(fig.axes[-1])
        fig.text(
            0.5,
            y_text,
            f"{frac}%",
            ha="center",
            va="bottom",
            fontsize=13,
            fontweight="bold",
        )

        # llamagen edited row
        ax_ll_ref = fig.add_subplot(gs[row_ptr, 0])
        show_image(
            ax_ll_ref,
            clean_ll,
            title=f"{frac}% | llamagen",
            bbox_px=bbox_px,
            bbox_color="yellow",
        )

        for c, mode in enumerate(modes, start=1):
            one_ll = row_ll[row_ll["mode"].astype(str) == mode]
            ax = fig.add_subplot(gs[row_ptr, c])
            if len(one_ll):
                edit_ll = load_img_float01(Path(one_ll["edit_path"].iloc[0]))
                show_image(
                    ax,
                    edit_ll,
                    title=mode,
                    bbox_px=bbox_px,
                    bbox_color="cyan",
                )
            else:
                ax.axis("off")

        row_ptr += 1

        # llamagen heatmaps
        ax_ll_label = fig.add_subplot(gs[row_ptr, 0])
        show_text_panel(ax_ll_label, heatmap_label)

        for c, mode in enumerate(modes, start=1):
            ax = fig.add_subplot(gs[row_ptr, c])
            if (frac, mode) in heatmaps_ll:
                show_heatmap(
                    ax,
                    heatmaps_ll[(frac, mode)],
                    title="",
                    bbox_px=bbox_px,
                    cmap="magma",
                    vmin=0.0,
                    vmax=vmax,
                )
            else:
                ax.axis("off")

        row_ptr += 1

        # vqgan edited row
        ax_vq_ref = fig.add_subplot(gs[row_ptr, 0])
        show_image(
            ax_vq_ref,
            clean_vq,
            title=f"{frac}% | vqgan",
            bbox_px=bbox_px,
            bbox_color="yellow",
        )

        for c, mode in enumerate(modes, start=1):
            one_vq = row_vq[row_vq["mode"].astype(str) == mode]
            ax = fig.add_subplot(gs[row_ptr, c])
            if len(one_vq):
                edit_vq = load_img_float01(Path(one_vq["edit_path"].iloc[0]))
                show_image(
                    ax,
                    edit_vq,
                    title=mode,
                    bbox_px=bbox_px,
                    bbox_color="cyan",
                )
            else:
                ax.axis("off")

        row_ptr += 1

        # vqgan heatmaps
        ax_vq_label = fig.add_subplot(gs[row_ptr, 0])
        show_text_panel(ax_vq_label, heatmap_label)

        for c, mode in enumerate(modes, start=1):
            ax = fig.add_subplot(gs[row_ptr, c])
            if (frac, mode) in heatmaps_vq:
                show_heatmap(
                    ax,
                    heatmaps_vq[(frac, mode)],
                    title="",
                    bbox_px=bbox_px,
                    cmap="magma",
                    vmin=0.0,
                    vmax=vmax,
                )
            else:
                ax.axis("off")

        row_ptr += 1

        # separator row between fraction blocks
        if fi != len(fractions) - 1:
            for c in range(ncols):
                ax_sep = fig.add_subplot(gs[row_ptr, c])
                ax_sep.axis("off")
                ax_sep.plot([0, 1], [0.5, 0.5], color="0.75", linewidth=1.2, transform=ax_sep.transAxes)
            row_ptr += 1

    fig.subplots_adjust(top=0.975, bottom=0.02, left=0.03, right=0.995)
    plt.show()


In [ ]:
DATASET_TO_SHOW = "imagenet_val"

for i, pack in enumerate(paired_blocks_by_dataset[DATASET_TO_SHOW]):
    print(f"\n=== {DATASET_TO_SHOW} | sample {i} | {pack['image_id']} ===")
    plot_fraction_blocks_comparison(
        pack["llamagen"],
        pack["vqgan"],
        heatmap_kind="rmse",
    )


In [ ]:
DATASET_TO_SHOW = "imagenetv2"

for i, pack in enumerate(paired_blocks_by_dataset[DATASET_TO_SHOW]):
    print(f"\n=== {DATASET_TO_SHOW} | sample {i} | {pack['image_id']} ===")
    plot_fraction_blocks_comparison(
        pack["llamagen"],
        pack["vqgan"],
    )


In [ ]:
DATASET_TO_SHOW = "imagenet_sketch"

for i, pack in enumerate(paired_blocks_by_dataset[DATASET_TO_SHOW]):
    print(f"\n=== {DATASET_TO_SHOW} | sample {i} | {pack['image_id']} ===")
    plot_fraction_blocks_comparison(
        pack["llamagen"],
        pack["vqgan"],
    )

## Analysis

In [ ]:
ANALYSIS_BUNDLE_NAMES = {
    "llamagen_imagenet_val": "1773180544_robustness_dataset_llamagen_h2_patch_token_edit_decoder_patchsweep10-25-50-75_seed0",
    "vqgan_imagenet_val": "1773180821_robustness_dataset_vqgan_h2_patch_token_edit_decoder_patchsweep10-25-50-75_seed0",
    "llamagen_imagenetv2": "1774520394_robustness_dataset_llamagen_h2_patch_token_edit_decoder_patchsweep10-25-50-75_seed0",
    "vqgan_imagenetv2": "1774519746_robustness_dataset_vqgan_h2_patch_token_edit_decoder_patchsweep10-25-50-75_seed0",
    "llamagen_sketch": "1773825918_robustness_dataset_llamagen_h2_patch_token_edit_decoder_patchsweep10-25-50-75_seed0",
    "vqgan_sketch": "1773825917_robustness_dataset_vqgan_h2_patch_token_edit_decoder_patchsweep10-25-50-75_seed0",
}


ANALYSIS_CACHE_ROOT = JUPYTER_ROOT / "analysis_cache"


In [ ]:
EXPECTED_OUTPUT_FILES = [
    "variants.csv.gz",
    "decoder_locality_psnr_ssim_all.csv.gz",
    "decoder_locality_lpips_all.csv.gz",
    "decoder_locality_psnr_ssim_lpips_merged.csv.gz",
    "manifest.json",
]

status_rows = []
for key, bundle_name in ANALYSIS_BUNDLE_NAMES.items():
    bundle_dir = ANALYSIS_CACHE_ROOT / bundle_name
    row = {
        "key": key,
        "bundle_name": bundle_name,
        "bundle_exists": bundle_dir.exists(),
    }
    for filename in EXPECTED_OUTPUT_FILES:
        row[filename] = (bundle_dir / filename).exists()
    status_rows.append(row)

status_df = pd.DataFrame(status_rows)
display(status_df)


### Fidelity Metrics

This section measures how much the edited decoder output differs from the clean decoder output. The comparison is always between the clean reconstruction and the edited reconstruction, not against the original input image.

We use three complementary fidelity metrics: PSNR, SSIM, and LPIPS, each computed on the full image and on the edited patch region. Together they capture pixel-level distortion, structural similarity, and perceptual change.

For each model, dataset, patch fraction, and edit mode, we aggregate the per-image fidelity scores and plot the mean with error bars. The error bars use the empirical standard deviation, so they show spread across images rather than uncertainty of the mean.

These plots are mainly descriptive. They show how fidelity changes as the token-edit mode becomes more disruptive and as the edited patch grows larger.


In [ ]:
def infer_model_from_key(key: str) -> str:
    if "llamagen" in key:
        return "llamagen"
    if "vqgan" in key:
        return "vqgan"
    raise ValueError(key)


def infer_dataset_from_key(key: str) -> str:
    if "imagenetv2" in key:
        return "imagenetv2"
    if "sketch" in key:
        return "sketch"
    return "imagenet_val"


loaded = {}

for key, bundle_name in ANALYSIS_BUNDLE_NAMES.items():
    bundle_dir = ANALYSIS_CACHE_ROOT / bundle_name
    loaded[key] = {
        "variants": pd.read_csv(bundle_dir / "variants.csv.gz"),
        "merged": pd.read_csv(bundle_dir / "decoder_locality_psnr_ssim_lpips_merged.csv.gz"),
    }

In [ ]:
master_df = pd.concat(
    [
        loaded[key]["merged"].assign(
            model=infer_model_from_key(key),
            dataset=infer_dataset_from_key(key),
        )
        for key in ANALYSIS_BUNDLE_NAMES
    ],
    ignore_index=True,
)

In [ ]:
summary_plot = (
    master_df
    .groupby(
        ["dataset", "fraction_label", "model", "mode"],
        as_index=False,
        observed=True,
    )
    .agg(
        n=("image_id", "count"),

        psnr_full_mean=("psnr_full", "mean"),
        psnr_full_std=("psnr_full", "std"),

        psnr_patch_mean=("psnr_patch", "mean"),
        psnr_patch_std=("psnr_patch", "std"),

        ssim_full_mean=("ssim_full", "mean"),
        ssim_full_std=("ssim_full", "std"),

        ssim_patch_mean=("ssim_patch", "mean"),
        ssim_patch_std=("ssim_patch", "std"),

        lpips_full_mean=("lpips_full", "mean"),
        lpips_full_std=("lpips_full", "std"),

        lpips_patch_mean=("lpips_patch", "mean"),
        lpips_patch_std=("lpips_patch", "std"),
    )
    .sort_values(["dataset", "fraction_label", "model", "mode"])
    .reset_index(drop=True)
)

for metric in [
    "psnr_full", "psnr_patch",
    "ssim_full", "ssim_patch",
    "lpips_full", "lpips_patch",
]:
    summary_plot[f"{metric}_sem"] = summary_plot[f"{metric}_std"] / np.sqrt(summary_plot["n"])

display(summary_plot.head(12))


In [ ]:
panel_grid = [
    [("psnr_full_mean",  "psnr_full_std",  "psnr"),  ("psnr_patch_mean",  "psnr_patch_std",  "psnr")],
    [("ssim_full_mean",  "ssim_full_std",  "ssim"),  ("ssim_patch_mean",  "ssim_patch_std",  "ssim")],
    [("lpips_full_mean", "lpips_full_std", "lpips"), ("lpips_patch_mean", "lpips_patch_std", "lpips")],
]

pretty_titles = {
    "psnr_full_mean": "PSNR full",
    "psnr_patch_mean": "PSNR patch",
    "ssim_full_mean": "SSIM full",
    "ssim_patch_mean": "SSIM patch",
    "lpips_full_mean": "LPIPS full",
    "lpips_patch_mean": "LPIPS patch",
}

colors = {
    "llamagen": "#5b74a8",
    "vqgan": "#c48f69",
}

baselines = {
    "imagenet_val": {
        "vqgan": {
            "psnr": {"mean": 19.649455, "std": 3.406491},
            "ssim": {"mean": 0.488608, "std": 0.175846},
            "lpips": {"mean": 0.285628, "std": 0.065720},
        },
        "llamagen": {
            "psnr": {"mean": 20.793682, "std": 3.184499},
            "ssim": {"mean": 0.558420, "std": 0.167186},
            "lpips": {"mean": 0.228152, "std": 0.058472},
        },
    },
    "imagenetv2": {
        "vqgan": {
            "psnr": {"mean": 19.831684, "std": 3.339825},
            "ssim": {"mean": 0.501436, "std": 0.171765},
            "lpips": {"mean": 0.288779, "std": 0.066414},
        },
        "llamagen": {
            "psnr": {"mean": 20.953198, "std": 3.132404},
            "ssim": {"mean": 0.571562, "std": 0.162310},
            "lpips": {"mean": 0.230828, "std": 0.058943},
        },
    },
    "sketch": {
        "vqgan": {
            "psnr": {"mean": 18.226843, "std": 4.386636},
            "ssim": {"mean": 0.695854, "std": 0.170250},
            "lpips": {"mean": 0.169755, "std": 0.077180},
        },
        "llamagen": {
            "psnr": {"mean": 20.258672, "std": 5.782762},
            "ssim": {"mean": 0.752088, "std": 0.150456},
            "lpips": {"mean": 0.127749, "std": 0.064468},
        },
    },
}

dataset_order = ["imagenet_val", "imagenetv2", "sketch"]
dataset_bar_source = {
    "imagenet_val": "imagenet_val",
    "imagenetv2": "imagenetv2",
    "sketch": "sketch",
}
dataset_titles = {
    "imagenet_val": "imagenet_val",
    "imagenetv2": "imagenetv2",
    "sketch": "sketch",
}

def metric_arrays(df, *, dataset, fraction, model, mean_col, err_col):
    sub = df[
        (df["dataset"] == dataset) &
        (df["fraction_label"].astype(str) == fraction) &
        (df["model"] == model)
    ]

    vals = []
    errs = []
    for mode in MODE_ORDER:
        one = sub[sub["mode"].astype(str) == mode]
        if len(one) == 0:
            vals.append(np.nan)
            errs.append(np.nan)
        else:
            vals.append(float(one[mean_col].iloc[0]))
            errs.append(float(one[err_col].iloc[0]))
    return np.asarray(vals, dtype=float), np.asarray(errs, dtype=float)

ylims = {}
for mean_col, err_col, fam in [item for row in panel_grid for item in row]:
    ymax = 0.0
    for dataset_display in dataset_order:
        for fraction in FRACTION_ORDER:
            for model in ["llamagen", "vqgan"]:
                vals, errs = metric_arrays(
                    summary_plot,
                    dataset=dataset_display,
                    fraction=fraction,
                    model=model,
                    mean_col=mean_col,
                    err_col=err_col,
                )
                arr = vals + errs
                arr = arr[np.isfinite(arr)]
                if len(arr):
                    ymax = max(ymax, float(arr.max()))

        for model in ["llamagen", "vqgan"]:
            base = baselines[dataset_display][model][fam]
            ymax = max(ymax, base["mean"] + base["std"])

    ylims[mean_col] = (0.0, ymax * 1.10)

for fraction in FRACTION_ORDER:
    fig, axes = plt.subplots(3, 6, figsize=(32, 15))
    fig.suptitle(
        f"Decoder-only edit fidelity by mode | patch {fraction}% | mean ± std\n",        
        fontsize=18,
        y=0.98,
    )

    bar_width = 0.35
    x = np.arange(len(MODE_ORDER))

    for i, row in enumerate(panel_grid):
        for j, (mean_col, err_col, fam) in enumerate(row):
            for k, dataset_display in enumerate(dataset_order):
                ax = axes[i, j * 3 + k]

                vals_ll, errs_ll = metric_arrays(
                    summary_plot,
                    dataset=dataset_display,
                    fraction=fraction,
                    model="llamagen",
                    mean_col=mean_col,
                    err_col=err_col,
                )
                vals_vq, errs_vq = metric_arrays(
                    summary_plot,
                    dataset=dataset_display,
                    fraction=fraction,
                    model="vqgan",
                    mean_col=mean_col,
                    err_col=err_col,
                )

                ax.bar(
                    x - bar_width / 2,
                    vals_ll,
                    width=bar_width,
                    yerr=errs_ll,
                    capsize=4,
                    color=colors["llamagen"],
                    label="llamagen" if (i == 0 and j == 0 and k == 0) else None,
                    alpha=0.75 if is_proxy else 0.95,
                    hatch="///" if is_proxy else None,
                )
                ax.bar(
                    x + bar_width / 2,
                    vals_vq,
                    width=bar_width,
                    yerr=errs_vq,
                    capsize=4,
                    color=colors["vqgan"],
                    label="vqgan" if (i == 0 and j == 0 and k == 0) else None,
                    alpha=0.75 if is_proxy else 0.95,
                    hatch="///" if is_proxy else None,
                )

                xmin = x[0] - 0.6
                xmax = x[-1] + 0.6
                for model in ["llamagen", "vqgan"]:
                    base = baselines[dataset_display][model][fam]
                    ax.fill_between(
                        [xmin, xmax],
                        [base["mean"] - base["std"], base["mean"] - base["std"]],
                        [base["mean"] + base["std"], base["mean"] + base["std"]],
                        color=colors[model],
                        alpha=0.10,
                    )
                    ax.axhline(
                        base["mean"],
                        color=colors[model],
                        linestyle="--",
                        linewidth=2,
                        alpha=0.9,
                        label=f"{model} baseline" if (i == 0 and j == 0 and k == 0) else None,
                    )

                ax.set_ylim(*ylims[mean_col])
                title = f"{pretty_titles[mean_col]} | {dataset_titles[dataset_display]}"
                if is_proxy:
                    title += " (bars from imagenetv2)"
                ax.set_title(title, fontsize=11)
                ax.set_xticks(x)
                ax.set_xticklabels(MODE_ORDER, rotation=30, ha="right")
                ax.grid(axis="y", linestyle="--", alpha=0.5)

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="center left", bbox_to_anchor=(1.01, 0.5), title="Legend")

    plt.tight_layout()
    plt.subplots_adjust(right=0.88)
    plt.show()


In [ ]:
baseline_rows = []
for dataset in ["imagenet_val", "imagenetv2", "imagenet_sketch"]:
    for model in ["llamagen", "vqgan"]:
        for fam in ["psnr", "ssim", "lpips"]:
            baseline_rows.append(
                {
                    "dataset": dataset,
                    "model": model,
                    "metric": fam.upper(),
                    "mean": baselines[dataset][model][fam]["mean"],
                    "std": baselines[dataset][model][fam]["std"],
                }
            )

baseline_df = pd.DataFrame(baseline_rows)

dataset_labels = {
    "imagenet_val": "ImageNet val",
    "imagenetv2": "ImageNetV2",
    "imagenet_sketch": "ImageNet-Sketch",
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
metric_order = ["PSNR", "SSIM", "LPIPS"]
x = np.arange(3)
bar_width = 0.35

for ax, metric_name in zip(axes, metric_order):
    sub = baseline_df[baseline_df["metric"] == metric_name]

    for offset, model in [(-bar_width / 2, "llamagen"), (bar_width / 2, "vqgan")]:
        sub_model = sub[sub["model"] == model].set_index("dataset").loc[["imagenet_val", "imagenetv2", "imagenet_sketch"]]
        ax.bar(
            x + offset,
            sub_model["mean"].to_numpy(),
            width=bar_width,
            yerr=sub_model["std"].to_numpy(),
            capsize=4,
            color=colors[model],
            alpha=0.95,
            label=model if metric_name == "PSNR" else None,
        )

    ax.set_title(f"{metric_name} reconstruction baseline", fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels([dataset_labels[d] for d in ["imagenet_val", "imagenetv2", "imagenet_sketch"]], rotation=20, ha="right")
    ax.grid(axis="y", linestyle="--", alpha=0.5)

axes[0].legend(title="Model")
fig.suptitle("Reconstruction baselines by dataset | me an ± std", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()


### Dataset Shift Heatmaps
To compare ImageNet validation and ImageNetV2 directly, we compute the difference
$
\Delta = \mu_{\mathrm{v2}} - \mu_{\mathrm{val}}
$
for each metric, patch fraction, mode, and model, where $\mu$ is the mean score over images.

Each heatmap cell shows this raw mean difference, and also a standardized version obtained by dividing by the pooled standard deviation of the two datasets. This lets us check not only whether the averages differ, but also whether the difference is large relative to the natural spread of the metric.

For each slice, we compute
$
\Delta = \mu_{\mathrm{v2}} - \mu_{\mathrm{val}},
$
and normalize it by
$
\sigma_{\mathrm{pooled}} = \sqrt{\frac{\sigma_{\mathrm{val}}^2 + \sigma_{\mathrm{v2}}^2}{2}}.
$
The annotation therefore reports both the raw shift and the shift relative to pooled variation.

Small raw shifts together with small standardized shifts indicate that dataset-level differences are weak compared with the within-dataset variability of the metric.

In [ ]:
delta_metric_specs = {
    "psnr_patch": ("psnr_patch_mean", "psnr_patch_std"),
    "ssim_patch": ("ssim_patch_mean", "ssim_patch_std"),
    "lpips_patch": ("lpips_patch_mean", "lpips_patch_std"),
}

delta_tables = {}

for metric_name, (mean_col, std_col) in delta_metric_specs.items():
    rows = []

    for model in ["llamagen", "vqgan"]:
        for fraction in ["10", "25", "50", "75"]:
            for mode in ["closest", "orthogonal", "random_uniform", "farthest"]:
                sub_val = summary_plot[
                    (summary_plot["dataset"] == "imagenet_val") &
                    (summary_plot["model"] == model) &
                    (summary_plot["fraction_label"].astype(str) == fraction) &
                    (summary_plot["mode"].astype(str) == mode)
                ]
                sub_v2 = summary_plot[
                    (summary_plot["dataset"] == "imagenetv2") &
                    (summary_plot["model"] == model) &
                    (summary_plot["fraction_label"].astype(str) == fraction) &
                    (summary_plot["mode"].astype(str) == mode)
                ]

                if len(sub_val) == 0 or len(sub_v2) == 0:
                    continue

                mean_val = float(sub_val[mean_col].iloc[0])
                mean_v2 = float(sub_v2[mean_col].iloc[0])
                std_val = float(sub_val[std_col].iloc[0])
                std_v2 = float(sub_v2[std_col].iloc[0])

                pooled_std = np.sqrt((std_val ** 2 + std_v2 ** 2) / 2.0)
                delta = mean_v2 - mean_val
                delta_sigma = delta / pooled_std if pooled_std > 0 else np.nan

                rows.append(
                    {
                        "metric": metric_name,
                        "model": model,
                        "fraction_label": fraction,
                        "mode": mode,
                        "mean_val": mean_val,
                        "mean_v2": mean_v2,
                        "delta": delta,
                        "std_val": std_val,
                        "std_v2": std_v2,
                        "pooled_std": pooled_std,
                        "delta_sigma": delta_sigma,
                    }
                )

    delta_tables[metric_name] = pd.DataFrame(rows)

display(delta_tables["lpips_patch"])

In [ ]:
def plot_dataset_delta_heatmaps(delta_df, metric_name, cmap="coolwarm"):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), constrained_layout=True)
    fig.suptitle(f"{metric_name}: imagenetv2 - imagenet_val", fontsize=16)

    for ax, model in zip(axes, ["llamagen", "vqgan"]):
        sub = delta_df[delta_df["model"] == model].copy()

        heat = (
            sub.pivot(index="fraction_label", columns="mode", values="delta")
            .reindex(index=["10", "25", "50", "75"], columns=["closest", "orthogonal", "random_uniform", "farthest"])
        )

        sigma = (
            sub.pivot(index="fraction_label", columns="mode", values="delta_sigma")
            .reindex(index=["10", "25", "50", "75"], columns=["closest", "orthogonal", "random_uniform", "farthest"])
        )

        vmax = np.nanmax(np.abs(heat.to_numpy()))
        vmax = max(vmax, 1e-8)

        im = ax.imshow(heat.to_numpy(dtype=float), cmap=cmap, vmin=-vmax, vmax=vmax, aspect="auto")
        ax.set_title(model)
        ax.set_xticks(range(len(heat.columns)))
        ax.set_xticklabels(heat.columns, rotation=30, ha="right")
        ax.set_yticks(range(len(heat.index)))
        ax.set_yticklabels([f"{x}%" for x in heat.index])

        for i in range(len(heat.index)):
            for j in range(len(heat.columns)):
                d = heat.iloc[i, j]
                s = sigma.iloc[i, j]
                if np.isfinite(d):
                    ax.text(
                        j,
                        i,
                        f"{d:+.3f}\n{s:+.2f}σ",
                        ha="center",
                        va="center",
                        fontsize=9,
                        color="black",
                    )

        for i in range(len(heat.index) + 1):
            ax.axhline(i - 0.5, color="white", linewidth=1, alpha=0.7)
        for j in range(len(heat.columns) + 1):
            ax.axvline(j - 0.5, color="white", linewidth=1, alpha=0.7)

    fig.colorbar(im, ax=axes, shrink=0.9, label="mean delta")
    plt.show()


plot_dataset_delta_heatmaps(delta_tables["psnr_patch"], "psnr_patch")
plot_dataset_delta_heatmaps(delta_tables["ssim_patch"], "ssim_patch")
plot_dataset_delta_heatmaps(delta_tables["lpips_patch"], "lpips_patch")


### Locality Metrics

This section measures how strongly the decoded image changes inside the edited patch and how much of that change leaks outside it. As in the fidelity section, the comparison is always between the clean reconstruction and the edited reconstruction.

Let
$
x_{\mathrm{clean}}, x_{\mathrm{edit}} \in [0,1]^{H \times W \times 3}
$
be the clean and edited decoded images, and let
$
B_{\mathrm{px}}
$
be the edited pixel-space patch. We first define a per-pixel change map
$
\Delta(u,v) = \left\|x_{\mathrm{edit}}(u,v) - x_{\mathrm{clean}}(u,v)\right\|_2,
$
so each pixel is assigned the magnitude of the decoder change at that location.

From this map, we compute the mean change inside and outside the edited region:
$
\mu_{\mathrm{in}} = \frac{1}{|B_{\mathrm{px}}|}\sum_{(u,v)\in B_{\mathrm{px}}}\Delta(u,v),
\qquad
\mu_{\mathrm{out}} = \frac{1}{|\overline{B_{\mathrm{px}}}|}\sum_{(u,v)\notin B_{\mathrm{px}}}\Delta(u,v).
$

The basic leakage ratio is then
$
\mathrm{leakage\ ratio} = \frac{\mu_{\mathrm{out}}}{\mu_{\mathrm{in}}}.
$
Smaller values indicate stronger locality: most of the change stays inside the edited patch. Larger values indicate that a larger fraction of the decoder response spreads outside the target region.

Because this ratio can become unstable when the inside-patch change is very small, we also use the analysis-facing version
$
\mathrm{leakage\ ratio\_safe} = \frac{\mu_{\mathrm{out}}}{\mu_{\mathrm{in}} + \varepsilon},
$
with a small $\varepsilon > 0$ for numerical stability. In practice, the median of this quantity is more reliable than the mean, since the mean can be dominated by rare outlier cases.

To capture not only average leakage but also extreme outside changes, we also compute high percentiles of the outside-patch change distribution:
$
\mathrm{outside\_p99}, \qquad \mathrm{outside\_p999}.
$
These tail statistics help detect localized but strong leakage even when the outside average remains modest.


In [ ]:
locality_summary_frames = []

for key, bundle_name in ANALYSIS_BUNDLE_NAMES.items():
    bundle_dir = ANALYSIS_CACHE_ROOT / bundle_name
    model = "llamagen" if "llamagen" in key else "vqgan"
    dataset = "imagenetv2" if "imagenetv2" in key else "imagenet_val"

    pixel_name = f"{bundle_name}__pixel_metrics.csv.gz"
    pixel_df = pd.read_csv(bundle_dir / pixel_name)

    pixel_df["fraction_label"] = pd.Categorical(
        pixel_df["fraction_label"].astype(str),
        categories=FRACTION_ORDER,
        ordered=True,
    )
    pixel_df["mode"] = pd.Categorical(
        pixel_df["mode"].astype(str),
        categories=MODE_ORDER,
        ordered=True,
    )

    summary = (
        pixel_df
        .groupby(
            ["fraction_label", "mode"],
            as_index=False,
            observed=True,
        )
        .agg(
            n=("image_id", "count"),

            inside_change_mean=("inside_change", "mean"),
            inside_change_std=("inside_change", "std"),

            outside_change_mean=("outside_change", "mean"),
            outside_change_std=("outside_change", "std"),

            leakage_ratio_safe_mean=("leakage_ratio_safe", "mean"),
            leakage_ratio_safe_std=("leakage_ratio_safe", "std"),
            leakage_ratio_safe_median=("leakage_ratio_safe", "median"),

            outside_p99_mean=("outside_p99", "mean"),
            outside_p99_std=("outside_p99", "std"),

            outside_p999_mean=("outside_p999", "mean"),
            outside_p999_std=("outside_p999", "std"),
        )
        .sort_values(["fraction_label", "mode"])
        .reset_index(drop=True)
    )

    summary["model"] = model
    summary["dataset"] = dataset
    locality_summary_frames.append(summary)

locality_summary = pd.concat(locality_summary_frames, ignore_index=True)
locality_summary = locality_summary[
    [
        "dataset",
        "fraction_label",
        "model",
        "mode",
        "n",
        "inside_change_mean",
        "inside_change_std",
        "outside_change_mean",
        "outside_change_std",
        "leakage_ratio_safe_mean",
        "leakage_ratio_safe_std",
        "leakage_ratio_safe_median",
        "outside_p99_mean",
        "outside_p99_std",
        "outside_p999_mean",
        "outside_p999_std",
    ]
]

display(locality_summary.head(12))


#### Absolute Outside-Patch Change

The leakage ratio is useful as a normalized concentration measure, but it does not directly reflect how much change appears outside the edited region in absolute terms.

To capture visible non-local effects more directly, we track the mean change outside the patch itself. Larger values here mean that more decoder activity is escaping the target region, regardless of how strong the change inside the patch is.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)
fig.suptitle("Mean outside-patch change across patch fractions", fontsize=16, y=0.98)

panel_order = [
    ("imagenet_val", "llamagen"),
    ("imagenet_val", "vqgan"),
    ("imagenetv2", "llamagen"),
    ("imagenetv2", "vqgan"),
]

for ax, (dataset, model) in zip(axes.flat, panel_order):
    sub = locality_summary[
        (locality_summary["dataset"] == dataset) &
        (locality_summary["model"] == model)
    ].copy()

    for mode in MODE_ORDER:
        one = sub[sub["mode"].astype(str) == mode].sort_values("fraction_label")
        ax.plot(
            one["fraction_label"].astype(str),
            one["outside_change_mean"],
            marker="o",
            linewidth=2,
            color=mode_colors[mode],
            label=mode,
        )

    for mode in MODE_ORDER:
        one = sub[sub["mode"].astype(str) == mode].sort_values("fraction_label")
        ax.plot(
            one["fraction_label"].astype(str),
            one["inside_change_mean"],
            marker="x",
            linestyle="--",
            linewidth=2,
            color=mode_colors[mode],
            label=mode,
        )

    ax.set_title(f"{dataset} | {model}", fontsize=11)
    ax.set_xlabel("patch fraction")
    ax.set_ylabel("mean outside change")
    ax.grid(True, linestyle="--", alpha=0.4)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="center left", bbox_to_anchor=(1.01, 0.5), title="mode")

plt.tight_layout()
plt.subplots_adjust(right=0.86)
plt.show()
